# SOCES Systematic Review Screener

**Similarity-Ordered Consecutive-Exclude Screening (SOCES)**  
Based on: *SOCES: Similarity-Ordered Consecutive-Exclude Stopping with Typed Probabilistic LLMs in Medical Systematic Reviews*

---

### What this notebook does

1. **Configure** — enter your OpenRouter API key and review objective  
2. **Upload** — load a `.bib` file (titles + abstracts required, max 2 000 records)  
3. **Embed** — compute `bge-m3` dense embeddings via OpenRouter and save to Parquet  
4. **Screen** — run SOCES: rank by cosine similarity → typed proposition scoring → consecutive-exclude stopping  
5. **Report** — display a PRISMA-trAIce–compatible decision table and summary  

### Requirements

```
pip install pybtex pandas pyarrow numpy requests tqdm ipywidgets
```

> **OpenRouter key** — obtain one at https://openrouter.ai  
> **Models used**  
> - Embeddings: `BAAI/bge-m3` via OpenRouter  
> - Screening: `typesafe/jev-1.13` via OpenRouter  

---

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

PACKAGES = ["pybtex", "pandas", "pyarrow", "numpy", "requests", "tqdm", "ipywidgets"]

for pkg in PACKAGES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

print("✓ All dependencies installed.")

## Cell 2 — Configuration

Enter your **OpenRouter API key** and the **systematic review objective** below.  
All other parameters are pre-set to the recommended SOCES defaults from the paper.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# REQUIRED — fill in these two fields
# ─────────────────────────────────────────────────────────────────

OPENROUTER_API_KEY = ""  # @param {type:"string"} # <── paste your OpenRouter key here

SR_OBJECTIVE = """ """                  # @param {type:"string_unrestricted"} # <── paste your systematic review objective here

# ─────────────────────────────────────────────────────────────────
# OPTIONAL — SOCES parameters (paper defaults shown)
# ─────────────────────────────────────────────────────────────────

N_CONSECUTIVE   = 20       # consecutive-exclude stopping window (n=20 recommended)
P_CUTOFF        = 0.50     # proposition inclusion threshold
MAX_BIB_RECORDS = 2_000    # hard cap on uploaded .bib size
ABSTRACT_CHARS  = 1_500    # characters of abstract fed to the screener
PARQUET_PATH     = "embeddings.parquet"   # output file for embedded corpus

# ─────────────────────────────────────────────────────────────────
# API endpoints (OpenRouter)
# ─────────────────────────────────────────────────────────────────

EMBED_MODEL     = "BAAI/bge-m3"
SCREEN_MODEL    = "typesafe/jev-1.13"
OR_BASE         = "https://openrouter.ai/api/v1"

import os
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

print("✓ Configuration saved.")
print(f"  Stopping window n = {N_CONSECUTIVE}")
print(f"  Inclusion threshold p ≥ {P_CUTOFF}")
print(f"  Max .bib records = {MAX_BIB_RECORDS}")

## Cell 3 — Upload and parse `.bib` file

**Requirements for the `.bib` file:**
- Each entry must have a **`title`** field  
- Each entry must have an **`abstract`** field  
- Maximum **2 000 entries** (enforced below)  
- UTF-8 encoding recommended  

Run the cell, then use the file picker to select your `.bib` file.

In [ ]:
import io, re
import pandas as pd
from pybtex.database.input import bibtex
from ipywidgets import FileUpload, Output
from IPython.display import display, clear_output

upload_widget = FileUpload(accept=".bib", multiple=False, description="Select .bib file")
out = Output()

# ── helpers ──────────────────────────────────────────────────────

def _strip_braces(s: str) -> str:
    """Remove LaTeX brace groups and clean whitespace."""
    return re.sub(r"[{}]", "", s).replace("\n", " ").strip()


def parse_bib_bytes(raw: bytes) -> pd.DataFrame:
    # Set pybtex's global strict mode to False to prevent errors on duplicates.
    import pybtex.errors
    pybtex.errors.set_strict_mode(False)

    # Set strict=False to allow parsing despite duplicate keys; custom handling follows.
    parser = bibtex.Parser(strict=False)
    bib_data = parser.parse_stream(io.StringIO(raw.decode("utf-8", errors="replace")))

    rows = []
    skipped_no_abstract = 0
    skipped_no_title    = 0
    seen_keys = set()
    skipped_duplicates = 0

    for key, entry in bib_data.entries.items():
        if key in seen_keys:
            skipped_duplicates += 1
            continue
        seen_keys.add(key)

        fields = {k.lower(): _strip_braces(str(v)) for k, v in entry.fields.items()}
        title    = fields.get("title", "").strip()
        abstract = fields.get("abstract", "").strip()

        if not title:
            skipped_no_title += 1
            continue
        if not abstract:
            skipped_no_abstract += 1
            continue

        rows.append({
            "key":      key,
            "title":    title,
            "abstract": abstract,
            "year":     fields.get("year", ""),
            "pmid":     fields.get("pmid", fields.get("pubmed", "")),
            "doi":      fields.get("doi", ""),
        })

    df = pd.DataFrame(rows)

    if len(df) > MAX_BIB_RECORDS:
        print(f"⚠ Truncating to {MAX_BIB_RECORDS} records (uploaded {len(df)}).")
        df = df.iloc[:MAX_BIB_RECORDS]

    return df, skipped_no_title, skipped_no_abstract, skipped_duplicates


def on_upload_change(change):
    with out:
        clear_output()
        uploaded = upload_widget.value
        if not uploaded:
            return
        name = list(uploaded.keys())[0]
        raw  = uploaded[name]["content"]

        global corpus_df
        corpus_df, skip_t, skip_a, skip_d = parse_bib_bytes(bytes(raw))

        print(f"✓ Parsed  : {name}")
        print(f"  Records with title + abstract : {len(corpus_df)}")
        if skip_t: print(f"  Skipped (no title)            : {skip_t}")
        if skip_a: print(f"  Skipped (no abstract)         : {skip_a}")
        if skip_d: print(f"  Skipped (duplicate key)       : {skip_d}")
        display(corpus_df[["key", "title", "year"]].head(5))


corpus_df = pd.DataFrame()   # populated after upload
upload_widget.observe(on_upload_change, names="value")

display(upload_widget, out)

In [ ]:
print(f"Total records in corpus_df: {len(corpus_df)}")
non_empty_abstracts = corpus_df[corpus_df['abstract'].str.strip() != '']
print(f"Records with non-empty abstracts: {len(non_empty_abstracts)}")

if not non_empty_abstracts.empty:
    print("\nFirst 5 records with non-empty abstracts:")
    display(non_empty_abstracts[['key', 'title', 'abstract']].head())
else:
    print("No records found with non-empty abstracts.")

## Cell 4 — Compute embeddings and save to Parquet

Embeds `title + abstract` for every record using **`BAAI/bge-m3`** via the OpenRouter embeddings endpoint.  
Results are saved to **`embeddings.parquet`** so you can re-run screening without re-embedding.

> Each record is embedded as a single string: `"<title>\n\n<abstract>"`  
> The review objective is embedded separately and stored in the same file as metadata.

In [ ]:
import json, time
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from tqdm.notebook import tqdm

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://github.com/soces-screening",
    "X-Title":       "SOCES Screening Notebook",
}


def embed_texts(texts: list[str], batch_size: int = 32) -> np.ndarray:
    """Return (N, D) float32 embedding matrix via OpenRouter bge-m3."""
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding batches"):
        batch = texts[i : i + batch_size]
        payload = {"model": EMBED_MODEL, "input": batch}
        for attempt in range(3):
            resp = requests.post(
                f"{OR_BASE}/embeddings", headers=HEADERS, json=payload, timeout=60
            )
            if resp.status_code == 200:
                break
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"  Rate limited — waiting {wait}s")
                time.sleep(wait)
            else:
                raise RuntimeError(f"Embedding API error {resp.status_code}: {resp.text[:200]}")
        data = resp.json()["data"]
        data.sort(key=lambda x: x["index"])
        all_vecs.extend([d["embedding"] for d in data])
    return np.array(all_vecs, dtype=np.float32)


if corpus_df.empty:
    raise RuntimeError("corpus_df is empty — run Cell 3 and upload a .bib file first.")

print(f"Embedding {len(corpus_df)} records …")

texts = [
    f"{row.title}\n\n{row.abstract}"
    for row in corpus_df.itertuples()
]

doc_embeddings = embed_texts(texts)

print("Embedding the review objective …")
# Strip retrospective results sections before embedding — matches the paper's
# _strip_results protocol so the query vector is purely prospective (no
# 'Results: we included N studies…' leakage into the similarity ranking).
import re as _re
objective_clean = _re.sub(
    r"(?i)(results?|findings?|conclusions?|main results?|we included \d+).*",
    "", SR_OBJECTIVE, flags=_re.DOTALL
).strip()
print(f"  Objective used for embedding ({len(objective_clean)} chars): {objective_clean[:120]}…")
obj_vec = embed_texts([objective_clean])[0]   # shape (D,)

# ── save to Parquet ───────────────────────────────────────────────
# Store each embedding vector as a list<float32> column.
embed_col = pa.array(
    [v.tolist() for v in doc_embeddings],
    type=pa.list_(pa.float32()),
)

table = pa.table(
    {
        "key":      pa.array(corpus_df["key"].tolist()),
        "title":    pa.array(corpus_df["title"].tolist()),
        "abstract": pa.array(corpus_df["abstract"].tolist()),
        "year":     pa.array(corpus_df["year"].tolist()),
        "pmid":     pa.array(corpus_df["pmid"].tolist()),
        "doi":      pa.array(corpus_df["doi"].tolist()),
        "embedding": embed_col,
    },
    metadata={
        b"embed_model":    EMBED_MODEL.encode(),
        b"sr_objective":   objective_clean.encode(),
        b"obj_embedding":  json.dumps(obj_vec.tolist()).encode(),
        b"abstract_chars": str(ABSTRACT_CHARS).encode(),
    },
)

pq.write_table(table, PARQUET_PATH, compression="snappy")

print(f"\n✓ Saved {len(corpus_df)} embedded records to '{PARQUET_PATH}'")
print(f"  Embedding dimension : {doc_embeddings.shape[1]}")
print(f"  File size           : {__import__('os').path.getsize(PARQUET_PATH) / 1024:.1f} KB")

## Cell 5 — Run SOCES screening

### Algorithm
1. Load Parquet, recover objective embedding from metadata  
2. Rank all records by **cosine similarity** (descending)  
3. Walk the ranked list and score each record with the **V7 `noul_relevance` proposition** via `typesafe/jev-1.13`  
4. **Include** if `p ≥ 0.50` (resets the exclude counter)  
5. **Stop** after `n = 20` consecutive exclusions — mark remaining records as AI-excluded  

### PRISMA-trAIce compliance
| Item | What SOCES provides |
|------|---------------------|
| M5   | One JSON log entry per candidate (`key`, `score`, `decision`) |
| M6   | V7 rubric text below, deposited as `v7_prompt_review.txt` |
| M7   | `p ≥ 0.50` cutoff + `n = 20` consecutive-exclude stopping |
| M8   | Model: `typesafe/jev-1.13`; embedding: `BAAI/bge-m3` |

In [ ]:
import json, time, pathlib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import requests
from tqdm.notebook import tqdm

# ── V7 noul_relevance rubric ─────────────────────────────────────────
V7_RUBRIC_TEMPLATE = """# PROPOSITION: noul_relevance

# CONTEXT

## Systematic review objective:
{sr_objective}

## Candidate title:
{candidate_title}

## Candidate abstract:
{candidate_abstract}

# QUESTION
Does this paper contribute relevant evidence, even indirectly,
to the systematic review objective stated above?

# TRUE IF (include the paper):
- The paper reports empirical data on the population, condition,
  intervention, or topic described in the review objective.
- The study design or subpopulation does not need to exactly match
  the review's primary methodology; indirect or background evidence
  qualifies.
- Surveillance data, observational cohorts, and multi-pathogen
  studies sharing vocabulary with the review objective are included.

# FALSE IF (exclude the paper):
- The paper's topic is entirely unrelated to the review objective.
- The paper contains no empirical data (e.g., editorial, commentary,
  opinion piece, letter without data, conference abstract only).
"""

# Split the rubric into context template and fixed instructions.
# state.text  → the CONTEXT block (filled per-record at runtime)
# instructions → the QUESTION + TRUE IF / FALSE IF block (fixed)
_q_idx = V7_RUBRIC_TEMPLATE.find("# QUESTION")
CONTEXT_TEMPLATE          = V7_RUBRIC_TEMPLATE[:_q_idx].strip()   # has {placeholders}
NOUL_RELEVANCE_INSTRUCTIONS = V7_RUBRIC_TEMPLATE[_q_idx:].strip()  # fixed text

# Deposit rubric text alongside the notebook
pathlib.Path("v7_prompt_review.txt").write_text(V7_RUBRIC_TEMPLATE)


# ── helpers ─────────────────────────────────────────

def cosine_similarity(a: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Return cosine similarities between vector a and matrix B (rows)."""
    a_norm = a / (np.linalg.norm(a) + 1e-12)
    b_norms = np.linalg.norm(B, axis=1, keepdims=True) + 1e-12
    return (B / b_norms) @ a_norm


def _strip_results(objective: str) -> str:
    """Remove retrospective results sections from the SR objective (data-leakage guard)."""
    return re.sub(
        r"(?i)(results?|findings?|conclusions?|main results?|we included \d+).*",
        "",
        objective,
        flags=re.DOTALL,
    ).strip()


_DEBUG_FIRST_RESPONSE = True   # prints first raw response; auto-disabled after

def score_record(sr_obj: str, title: str, abstract: str) -> float:
    """Call typesafe/jev-1.13 via /api/alpha/decisions.

    Payload shape (confirmed from API error message):
      state.text  = filled CONTEXT block (objective + title + abstract)
      questions   = {"noul_relevance": {"type": "noul", "instructions": <QUESTION+criteria>}}

    Response shape: {"decisions": [{"name": "noul_relevance", "score": 0.94, ...}]}
    """
    global _DEBUG_FIRST_RESPONSE

    # Fill the CONTEXT block only (objective / title / abstract placeholders)
    context_text = CONTEXT_TEMPLATE.format(
        sr_objective=sr_obj,
        candidate_title=title,
        candidate_abstract=abstract[:ABSTRACT_CHARS],
    )

    payload = {
        "model": SCREEN_MODEL,
        "state": {"text": context_text},
        "questions": {
            "noul_relevance": {
                "type": "noul",
                "instructions": NOUL_RELEVANCE_INSTRUCTIONS,
            }
        },
    }

    for attempt in range(3):
        resp = requests.post(
            "https://openrouter.ai/api/alpha/decisions",
            headers=HEADERS,
            json=payload,
            timeout=30,
        )
        if resp.status_code == 200:
            break
        if resp.status_code == 429:
            time.sleep(2 ** attempt)
        else:
            raise RuntimeError(f"Screening API error {resp.status_code}: {resp.text}")

    body = resp.json()

    # Print first response in full so the response shape is visible
    if _DEBUG_FIRST_RESPONSE:
        print("\n── DEBUG: first raw decisions response ──")
        print(json.dumps(body, indent=2)[:1500])
        print("─────────────────────────────────────────\n")
        _DEBUG_FIRST_RESPONSE = False

    # ── Parse: body["answers"]["noul_relevance"]["noul"] ─────────
    # Confirmed response shape:
    # {"answers": {"noul_relevance": {"type": "noul", "noul": 0.89}}}
    try:
        p = float(body["answers"]["noul_relevance"]["noul"])
        if 0.0 <= p <= 1.0:
            return p
    except (KeyError, TypeError, ValueError):
        pass

    # Last resort: print body so response shape is visible
    print(f"  WARN: unexpected response shape: {str(body)[:400]}")
    return 0.0


# ── load Parquet ─────────────────────────────────────────
import re

table   = pq.read_table(PARQUET_PATH)
meta    = table.schema.metadata
df      = table.to_pandas()

obj_vec = np.array(json.loads(meta[b"obj_embedding"]), dtype=np.float32)
sr_obj_clean = _strip_results(SR_OBJECTIVE.strip())

# Stack embeddings → (N, D)
E = np.stack(df["embedding"].tolist()).astype(np.float32)

# ── Stage 1: cosine-rank ──────────────────────────────────
similarities = cosine_similarity(obj_vec, E)
ranked_idx   = np.argsort(-similarities)   # descending

print(f"✓ Ranked {len(df)} records by cosine similarity to the review objective.")
print(f"  Top similarity score: {similarities[ranked_idx[0]]:.4f}")
print(f"  Median similarity   : {np.median(similarities):.4f}")

# ── Stage 2: SOCES screening ───────────────────────────
results = []
consecutive_excludes = 0
stopped_at_rank      = None

print(f"\nRunning SOCES (n={N_CONSECUTIVE}, p≥{P_CUTOFF}) …")

for rank, idx in enumerate(tqdm(ranked_idx, desc="Screening")):
    row = df.iloc[idx]

    if stopped_at_rank is not None:
        # Beyond stopping point ─ AI excluded without scoring
        results.append({
            "rank":       rank,
            "key":        row["key"],
            "pmid":       row["pmid"],
            "title":      row["title"],
            "similarity": float(similarities[idx]),
            "score":      None,
            "decision":   False,
            "source":     "ai_stopped",
        })
        continue

    # Score this record
    p = score_record(sr_obj_clean, row["title"], row["abstract"])
    include = p >= P_CUTOFF

    results.append({
        "rank":       rank,
        "key":        row["key"],
        "pmid":       row["pmid"],
        "title":      row["title"],
        "similarity": float(similarities[idx]),
        "score":      round(p, 4),
        "decision":   include,
        "source":     "ai_scored",
    })

    if include:
        consecutive_excludes = 0
    else:
        consecutive_excludes += 1
        if consecutive_excludes >= N_CONSECUTIVE:
            stopped_at_rank = rank

results_df = pd.DataFrame(results)
results_df.to_csv("results.csv", index=False)

# ── summary ─────────────────────────────────────────
n_total       = len(results_df)
n_scored      = (results_df["source"] == "ai_scored").sum()
n_stopped     = (results_df["source"] == "ai_stopped").sum()
n_included    = results_df["decision"].sum()
n_ai_excluded = (~results_df["decision"]).sum()
calls_saved   = n_stopped
calls_saved_pct = 100 * calls_saved / n_total if n_total else 0

print(f"\n{'-─'*27}")
print(f"  Total candidates          : {n_total}")
print(f"  Records scored by model   : {n_scored}")
print(f"  Records stopped (unscored): {n_stopped}  ({calls_saved_pct:.1f}% calls saved)")
print(f"  ──")
print(f"  Included (p ≥ {P_CUTOFF})       : {n_included}")
print(f"  Excluded by AI            : {n_ai_excluded}")
if stopped_at_rank:
    print(f"  Stopping triggered at rank: {stopped_at_rank}")
else:
    print(f"  Stopping: not triggered (full corpus screened)")
print(f"{'-─'*27}")
print(f"\n✓ Decision log saved to 'results.csv'")

## Cell 6 — PRISMA-trAIce results display

Shows:
- Included records forwarded for human review  
- AI exclusion breakdown  
- PRISMA-trAIce flow node counts (M5 compliant)  
- Score distribution summary

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# Load results if running standalone
if "results_df" not in dir() or results_df.empty:
    results_df = pd.read_csv("results.csv")

scored    = results_df[results_df["source"] == "ai_scored"]
included  = results_df[results_df["decision"] == True]
ai_excl   = results_df[results_df["decision"] == False]
by_score  = ai_excl[ai_excl["source"] == "ai_scored"]
by_stop   = ai_excl[ai_excl["source"] == "ai_stopped"]

# ── PRISMA-trAIce flow block ──────────────────────────────────────
prisma_html = f"""
<style>
  .prisma-box {{
    border: 2px solid #3b82d4; border-radius: 6px;
    padding: 12px 18px; margin: 6px 0;
    font-family: -apple-system, "Segoe UI", sans-serif;
    font-size: 14px; line-height: 1.6;
    background: #f0f6ff;
  }}
  .prisma-excl {{
    border-color: #e57373; background: #fff0f0;
  }}
  .prisma-human {{
    border-color: #43a047; background: #f0fff4;
  }}
  .prisma-label {{
    font-weight: 700; color: #1f2328;
  }}
  .prisma-sub {{ color: #57606a; font-size: 12px; }}
</style>
<h3 style="font-family:sans-serif; margin-bottom:4px">PRISMA-trAIce Flow — Title &amp; Abstract Screening</h3>
<p style="font-family:sans-serif; font-size:12px; color:#57606a">
  Models: embed=<code>{EMBED_MODEL}</code> · screen=<code>{SCREEN_MODEL}</code> ·
  Cutoff p≥{P_CUTOFF} · Stopping n={N_CONSECUTIVE}
</p>

<div class="prisma-box">
  <span class="prisma-label">Records identified from database search</span><br>
  <span class="prisma-sub">n = {len(results_df)}</span>
</div>

<div style="margin-left:30px">
  <div class="prisma-box prisma-excl">
    <span class="prisma-label">Records excluded by AI</span> &nbsp;
    <span style="font-size:16px; font-weight:700">n = {len(ai_excl)}</span><br>
    <span class="prisma-sub">
      • Scored p &lt; {P_CUTOFF} &nbsp;&nbsp; n = {len(by_score)}<br>
      • Unscored after consecutive-exclude stopping (n={N_CONSECUTIVE}) &nbsp;&nbsp; n = {len(by_stop)}
    </span>
  </div>

  <div class="prisma-box prisma-human">
    <span class="prisma-label">Records forwarded for human review</span> &nbsp;
    <span style="font-size:16px; font-weight:700">n = {len(included)}</span><br>
    <span class="prisma-sub">All scored p ≥ {P_CUTOFF} — screener must apply full inclusion/exclusion criteria</span>
  </div>
</div>
"""

display(HTML(prisma_html))

# ── Score distribution ─────────────────────────────────────────────
print("\n── Score distribution (scored records only) ──")
s = scored["score"].dropna()
if len(s):
    print(f"  Mean   : {s.mean():.3f}")
    print(f"  Median : {s.median():.3f}")
    print(f"  Pct ≥ 0.50 : {100*(s >= 0.50).mean():.1f}%")
    print(f"  Pct ≥ 0.75 : {100*(s >= 0.75).mean():.1f}%")
    print(f"  Pct ≥ 0.90 : {100*(s >= 0.90).mean():.1f}%")

# ── Included records table ─────────────────────────────────────────
print(f"\n── Included records (first 20 of {len(included)}) ──")
display(
    included[["rank", "title", "score", "similarity", "pmid"]]
    .head(20)
    .reset_index(drop=True)
    .style.format({"score": "{:.3f}", "similarity": "{:.3f}"})
    .background_gradient(subset=["score"], cmap="Greens")
)

## Cell 7 — Export full decision log

Exports `results.csv` (already written) and a `results_included.csv` with only the records forwarded for human review.  
Both files satisfy **PRISMA-trAIce item M5** — one row per candidate, structured fields, zero reconstruction needed.

In [ ]:
import json, pathlib
import pandas as pd

if "results_df" not in dir() or results_df.empty:
    results_df = pd.read_csv("results.csv")

# Full log — all candidates, one row each
results_df.to_csv("results.csv", index=False)

# Included set forwarded to human screener
included = results_df[results_df["decision"] == True].copy()
included.to_csv("results_included.csv", index=False)

# Machine-readable summary (mirrors PRISMA-trAIce flow node counts)
summary = {
    "sr_objective_preview":      SR_OBJECTIVE.strip()[:200],
    "embed_model":                EMBED_MODEL,
    "screen_model":               SCREEN_MODEL,
    "p_cutoff":                   P_CUTOFF,
    "n_consecutive":              N_CONSECUTIVE,
    "total_candidates":           int(len(results_df)),
    "records_scored":             int((results_df["source"] == "ai_scored").sum()),
    "records_stopped_unscored":   int((results_df["source"] == "ai_stopped").sum()),
    "included_for_human_review":  int(results_df["decision"].sum()),
    "excluded_by_ai":             int((~results_df["decision"]).sum()),
    "calls_saved_pct":            round(
        100 * (results_df["source"] == "ai_stopped").sum() / len(results_df), 2
    ) if len(results_df) else 0,
}

pathlib.Path("summary.json").write_text(json.dumps(summary, indent=2))

print("Exported files:")
for fname in ["results.csv", "results_included.csv", "summary.json", "v7_prompt_review.txt"]:
    size_kb = pathlib.Path(fname).stat().st_size / 1024
    print(f"  {fname:<30} {size_kb:6.1f} KB")

print("\n✓ PRISMA-trAIce artefacts ready.")
print(f"  M5 — decision log        : results.csv")
print(f"  M6 — proposition rubric  : v7_prompt_review.txt")
print(f"  M7 — thresholds          : p ≥ {P_CUTOFF}, n = {N_CONSECUTIVE}")
print(f"  M8 — tool versions       : {SCREEN_MODEL} / {EMBED_MODEL}")

---

## Appendix — Sensitivity sweep across stopping parameters

*(Optional)* Re-score at multiple values of `n` using the already-scored records in `results.csv`.  
No additional API calls are made — this replays the stopping logic across the saved scores.

In [ ]:
import pandas as pd
import numpy as np

if "results_df" not in dir() or results_df.empty:
    results_df = pd.read_csv("results.csv")

# Only replay over records that were actually scored (have a numeric score)
scored_df = results_df[results_df["score"].notna()].copy()
scored_df = scored_df.sort_values("rank").reset_index(drop=True)

sweep_ns = [5, 10, 20, 30, 50, 100, len(scored_df)]  # last entry = unstopped

rows = []
for n_val in sweep_ns:
    consec = 0
    included_keys = []
    calls = 0
    stopped = False

    for _, rec in scored_df.iterrows():
        if stopped:
            break
        calls += 1
        if rec["score"] >= P_CUTOFF:
            included_keys.append(rec["key"])
            consec = 0
        else:
            consec += 1
            if consec >= n_val:
                stopped = True

    rows.append({
        "n": n_val if n_val < len(scored_df) else "∞ (unstopped)",
        "included": len(included_keys),
        "calls": calls,
        "calls_saved": len(scored_df) - calls,
        "calls_saved_pct": f"{100*(len(scored_df)-calls)/len(scored_df):.1f}%",
        "early_stopped": stopped,
    })

sweep_summary = pd.DataFrame(rows)
print("Stopping parameter sweep (using saved scores — no additional API calls):")
display(sweep_summary.style.set_caption("SOCES n-sweep"))
sweep_summary.to_csv("sweep.csv", index=False)
print("\n✓ Sweep saved to 'sweep.csv'")